# Training report — `cap4500_yolov8s_v2-2`

**v2 of the Second Vision obstacle detector: YOLOv8s, 14 classes, 640 px**, trained on the
cap-4500 condition and evaluated on the frozen test split.

v2 exists to answer one question: **how much of v1's shortfall was a model problem and how much
was an annotation problem?** v1 reached mAP@0.5 = 0.6695 against a pre-registered target of 0.70
(**DEC-122**, which set the 0.70 criterion as documentation-only and named Pole and Potholes as
expected-weak in advance). **DEC-124** diagnosed the shortfall as pure recall rather than class
confusion, driven by instance density and by Open Images' non-exhaustive annotation — 359,760
downloaded boxes were discarded by the converter's per-folder filter, including 8,489 Chairs
sitting unlabelled in the `tables` folder.

**v1 and v2 share the same images, the same splits and the same hyperparameters. Only the labels
differ.** That is what makes the comparison controlled.

What changed in the labels:

| decision | change |
|---|---|
| **DEC-126** | Shelf dropped, 15 → 14 classes; every id above 5 shifted down by one |
| **DEC-127** | converter widened to a schema-wide filter; Person gains Man/Woman/Boy/Girl, Chairs gains Couch/Stool, Tables gains Desk (+53% annotations) |
| **DEC-129** | ExDark `Bus` → Vehicle, applied only to the 330 already-pooled images (+413 boxes) |
| **DEC-130** | CrowdHuman id-shift repaired — 56 boxes had been written with 14-class ids into the 15-class tree |
| **DEC-131** | hovyc labels restored; a stray bicycle box removed from a pothole image |
| **DEC-132** | geometric dedup at IoU ≥ 0.90 — 2,225 duplicate boxes corpus-wide, 798 of them in the training pool |
| **DEC-133** | Open Images hierarchy duplicates — 824 removed (728 hierarchy collapses such as Desk+Table, 96 nested), 31 genuine overlaps protected |

**DEC-134** governs how the result may be reported: a 14-class mAP must not be compared against a
15-class mAP. Section 3 works through why.

Everything below is read from artefacts on disk — `results.csv`, `args.yaml`, `provenance.json`,
`RESULTS.md` and the evaluation outputs — so the report cannot drift from the run it describes.

In [ ]:
from pathlib import Path
import json, re

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ROOT = Path.cwd()
while not (ROOT / "config" / "classes.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RUN    = ROOT / "runs" / "detect" / "cap4500_yolov8s_v2-2"   # v2
RUN_V1 = ROOT / "runs" / "detect" / "cap4500_yolov8s"        # v1, for comparison
FIGS   = ROOT / "docs" / "figures" / "v2"                    # NOT docs/figures - v1 owns that
FIGS.mkdir(parents=True, exist_ok=True)

df      = pd.read_csv(RUN / "eval_fp32_test_per_class.csv")
summary = json.loads((RUN / "eval_fp32_test_summary.json").read_text())
prov    = json.loads((RUN / "provenance.json").read_text())
hist    = pd.read_csv(RUN / "results.csv")
hist.columns = [c.strip() for c in hist.columns]

df_v1      = pd.read_csv(RUN_V1 / "eval_fp32_test_per_class.csv")
summary_v1 = json.loads((RUN_V1 / "eval_fp32_test_summary.json").read_text())

TARGET = summary["target_map50"]
print(f"v2 artefacts : {RUN.relative_to(ROOT)}")
print(f"v1 artefacts : {RUN_V1.relative_to(ROOT)}")
print(f"figures      : {FIGS.relative_to(ROOT)}")

## 1. Provenance

The run was produced on the pod and the container disk has since been wiped, so
`provenance.json` and `pip_freeze.txt` inside the run directory are the **only** surviving record
of the environment — this is exactly why **DEC-120** designates them, not
`requirements-train.txt`, as authoritative.

In [ ]:
import yaml
args = yaml.safe_load((RUN / "args.yaml").read_text())

rows = [
    ("run name",      RUN.name),
    ("finished",      prov["finished"]),
    ("git SHA",       prov["git_sha"][:12]),
    ("data.yaml",     prov["data_yaml"]),
    ("classes (nc)",  prov["data_yaml_contents"]["nc"]),
    ("loggers armed", ", ".join(k for k, v in prov["loggers"].items() if v) or "none"),
    ("smoke run?",    prov["smoke"]),
]
for k, v in rows:
    print(f"  {k:<16} {v}")

freeze = (RUN / "pip_freeze.txt").read_text().splitlines()
key = [l for l in freeze if l.lower().startswith(("torch==", "ultralytics==", "numpy==", "opencv"))]
print("\n  key packages:")
for l in key:
    print(f"    {l}")

print("\n  class map actually trained against:")
names = prov["data_yaml_contents"]["names"]
print("    " + ", ".join(f"{k}:{v}" for k, v in sorted(names.items(), key=lambda kv: int(kv[0]))))
assert "Shelf" not in names.values(), "Shelf must be gone in v2 (DEC-126)"
print("\n  Shelf absent from the trained schema - DEC-126 holds.")

## 2. Configuration — identical to v1 by construction

The controlled comparison only means anything if nothing but the labels moved. Rather than
assert that, diff the two `args.yaml` files and show what actually differs.

In [ ]:
args_v1 = yaml.safe_load((RUN_V1 / "args.yaml").read_text())

keys = sorted(set(args) | set(args_v1))
diffs = [(k, args_v1.get(k, "<absent>"), args.get(k, "<absent>"))
         for k in keys if args_v1.get(k, "<absent>") != args.get(k, "<absent>")]

print(f"{len(keys)} hyperparameters compared; {len(diffs)} differ\n")
print(f"  {'key':<16} {'v1':<34} {'v2'}")
print("  " + "-" * 74)
for k, a, b in diffs:
    print(f"  {k:<16} {str(a)[-32:]:<34} {str(b)[-32:]}")

EXPECTED = {"data", "name", "save_dir", "project"}
unexpected = {k for k, _, _ in diffs} - EXPECTED
print("\n  " + ("CLEAN - only path/name keys differ, so the schedule is identical."
                 if not unexpected else f"UNEXPECTED DIFFERENCES: {sorted(unexpected)}"))

sched = ["model", "imgsz", "epochs", "batch", "patience", "optimizer", "lr0", "seed"]
print("\n  shared schedule:")
for k in sched:
    print(f"    {k:<12} {args.get(k)}")

## 3. The controlled comparison — and why only one number is valid

**DEC-134** is the rule this section enforces: **a 14-class mAP may not be compared against a
15-class mAP.** Dropping Shelf removes a class that scored 0.19 from the average, so the mean
rises even if no detection improved. Three v1 numbers are in circulation and only one is a fair
comparator:

| number | what it is | usable as v2's baseline? |
|---|---|---|
| 0.6695 | published v1, 15 classes, **v1's own sparse labels** | **No** — inflated by boxes that were missing from the ground truth, so the model was never penalised for missing them |
| 0.6417 | v1 re-measured on corrected labels, still 15 classes, stale, measured on a laptop | **No** — carries Shelf, and not the same environment |
| **0.6825** | **v1 re-scored on the pod against the current v2a labels, averaged over the same 14 classes** | **Yes** |

`post_training_v2.sh` computed the third by running v1's `best.pt` against `final_cap4500_v2a`
on the training pod itself. That is the only apples-to-apples figure, and it is what the delta
below uses.

In [ ]:
raw = (ROOT / "runs" / "detect" / "RESULTS.md").read_text(errors="replace")
raw = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", raw)          # strip the tee'd progress-bar ANSI

recs = []
for line in raw.splitlines():
    if not line.startswith("|"):
        continue
    p = [c.strip() for c in line.strip("|").split("|")]
    if len(p) != 3 or p[0] in ("class",) or set(p[0]) <= set("- "):
        continue
    def num(x):
        try:    return float(x)
        except ValueError: return float("nan")
    recs.append({"class": p[0], "v2": num(p[1]), "v1_on_v2a": num(p[2])})

cmp = pd.DataFrame(recs)
shared = cmp[cmp["class"] != "Shelf"].copy()
shared["delta"] = shared["v2"] - shared["v1_on_v2a"]

v2_14  = shared["v2"].mean()
v1_14  = shared["v1_on_v2a"].mean()
v1_15  = cmp["v1_on_v2a"].mean()
shelf  = float(cmp.loc[cmp["class"] == "Shelf", "v1_on_v2a"].iloc[0])

print(f"  classes compared                     {len(shared)}")
print(f"  v2, 14-class mean mAP@0.5            {v2_14:.4f}")
print(f"  v1 re-baselined, 14-class mean       {v1_14:.4f}   <- the number to beat")
print(f"  delta (v2 - v1, both 14-class)       {v2_14 - v1_14:+.4f}")
print()
print(f"  v1 re-baselined, 15-class mean       {v1_15:.4f}   (includes Shelf at {shelf:.4f})")
print(f"  Shelf's drag on the 15-class mean    {v1_14 - v1_15:+.4f}")
print(f"  published v1 (15 cls, v1 labels)     0.6695   inflated by missing annotations")
print()
print(f"  ultralytics' own map50 for v2        {summary['map50']:.5f}")
print("    (differs in the 4th decimal from the per-class mean above - ultralytics averages")
print("     internally over classes present in the split; both are reported for transparency)")

print(f"\n  REPORTABLE RESULT: v2 {v2_14:.4f} vs v1 {v1_14:.4f} = {v2_14 - v1_14:+.4f}")
print(f"  Do NOT report {v2_14:.4f} against 0.6497 or 0.6695 - that mixes 14- and 15-class means")
print(f"  and would overstate the gain by {(v1_14 - v1_15):.4f} (DEC-134).")

### 3a. Per-class movement

A dumbbell is the right form here — the question is *before → after per item*, not the levels
themselves. One hue, two shades: v1 light, v2 dark. Classes are ordered by v2 score.

In [ ]:
# --- chart style, shared by every figure below -------------------------------
# Light surface only: a notebook figure has one rendering context, so this is a
# deliberate single-mode commitment rather than an un-themed oversight.
SURFACE   = "#fcfcfb"
INK       = "#0b0b0b"
INK_2     = "#52514e"
GRID      = "#e3e2de"
BLUE      = "#2a78d6"   # categorical slot 1 / sequential hue
BLUE_LT   = "#9ec5f4"   # same ramp, lighter step - "before"
BLUE_DK   = "#1c5cab"   # same ramp, darker step - "after"
ORANGE    = "#eb6834"   # categorical slot 2
RED       = "#e34948"   # diverging pole (negative)

def style_axes(ax, xlabel=None, ylabel=None, title=None):
    ax.set_facecolor(SURFACE)
    ax.figure.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK_2, labelsize=9, length=0)
    if xlabel: ax.set_xlabel(xlabel, color=INK_2, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=INK_2, fontsize=9)
    if title:  ax.set_title(title, color=INK, fontsize=11, loc="left", pad=12)
    return ax

d = shared.sort_values("v2").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 6))
y = range(len(d))
ax.hlines(y, d["v1_on_v2a"], d["v2"], color=GRID, lw=2, zorder=1)
ax.scatter(d["v1_on_v2a"], y, s=58, color=BLUE_LT, zorder=2, label="v1 (re-baselined)",
           edgecolor=SURFACE, linewidth=1.5)
ax.scatter(d["v2"], y, s=58, color=BLUE_DK, zorder=3, label="v2",
           edgecolor=SURFACE, linewidth=1.5)
for i, r in d.iterrows():                                   # selective direct labels
    if abs(r["delta"]) >= 0.02:
        ax.annotate(f"{r['delta']:+.3f}", (max(r["v2"], r["v1_on_v2a"]) + 0.012, i),
                    va="center", fontsize=8, color=INK_2)
ax.set_yticks(list(y)); ax.set_yticklabels(d["class"], fontsize=9, color=INK)
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.set_xlim(0.30, 1.0)
style_axes(ax, xlabel="mAP@0.5 (test split, both scored on current labels)",
           title="v1 → v2 per class  ·  deltas labelled where |Δ| ≥ 0.02")
ax.legend(frameon=False, loc="lower right", fontsize=9, labelcolor=INK_2)
fig.tight_layout(); fig.savefig(FIGS / "per_class_dumbbell_v1_v2.png", dpi=160,
                                facecolor=SURFACE); plt.show()

In [ ]:
# Diverging bars: the data's job here is polarity (who gained, who lost).
d = shared.sort_values("delta").reset_index(drop=True)
colors = [BLUE if v >= 0 else RED for v in d["delta"]]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(range(len(d)), d["delta"], color=colors, height=0.62)
ax.axvline(0, color=INK_2, lw=1)
ax.set_yticks(range(len(d))); ax.set_yticklabels(d["class"], fontsize=9, color=INK)
for i, v in enumerate(d["delta"]):
    ax.annotate(f"{v:+.3f}", (v + (0.004 if v >= 0 else -0.004), i),
                va="center", ha="left" if v >= 0 else "right", fontsize=8, color=INK_2)
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.set_xlim(min(d["delta"]) - 0.045, max(d["delta"]) + 0.055)
style_axes(ax, xlabel="Δ mAP@0.5  (v2 − v1, both on current labels)",
           title="Where v2 gained and where it lost")
fig.tight_layout(); fig.savefig(FIGS / "per_class_delta_v2.png", dpi=160,
                                facecolor=SURFACE); plt.show()

gained = shared[shared["delta"] > 0].sort_values("delta", ascending=False)
lost   = shared[shared["delta"] < 0].sort_values("delta")
print(f"  gained: {len(gained)}/{len(shared)} classes   lost: {len(lost)}/{len(shared)}")
print(f"  largest gain  {gained.iloc[0]['class']:<12} {gained.iloc[0]['delta']:+.4f}")
print(f"  largest loss  {lost.iloc[0]['class']:<12} {lost.iloc[0]['delta']:+.4f}")
print(f"  sum of losses {lost['delta'].sum():+.4f} vs sum of gains {gained['delta'].sum():+.4f}")

## 4. Where the gain came from — the annotation recovery

The images are identical between v1 and v2, so any change in ground-truth instance count is
purely the label work of **DEC-127** and **DEC-129 to DEC-133**. If the annotation-defect
diagnosis in **DEC-124** is right, the classes that gained the most ground truth should be the
classes that gained the most mAP.

In [ ]:
gt = (df[["class", "instances"]].rename(columns={"instances": "v2_gt"})
        .merge(df_v1[["class", "instances"]].rename(columns={"instances": "v1_gt"}),
               on="class", how="left"))
gt["gt_delta"] = gt["v2_gt"] - gt["v1_gt"]
gt["gt_pct"]   = 100 * gt["gt_delta"] / gt["v1_gt"]
gt = gt.merge(shared[["class", "delta"]], on="class", how="left")

print(f"  {'class':<12} {'v1 GT':>7} {'v2 GT':>7} {'Δ GT':>7} {'Δ GT %':>8} {'Δ mAP':>8}")
print("  " + "-" * 54)
for _, r in gt.sort_values("gt_pct", ascending=False).iterrows():
    print(f"  {r['class']:<12} {r['v1_gt']:>7.0f} {r['v2_gt']:>7.0f} "
          f"{r['gt_delta']:>+7.0f} {r['gt_pct']:>+7.1f}% {r['delta']:>+8.4f}")

tot_v1_14 = df_v1[df_v1['class'] != 'Shelf']['instances'].sum()
print(f"\n  14-class GT total   v1 {tot_v1_14:,}  ->  v2 {df['instances'].sum():,} "
      f"({100*(df['instances'].sum()-tot_v1_14)/tot_v1_14:+.1f}%)")
print(f"  Shelf removed from v1's 15-class total: {int(df_v1[df_v1['class']=='Shelf']['instances'].iloc[0]):,} boxes")

from scipy.stats import spearmanr
ok = gt.dropna(subset=["gt_pct", "delta"])
rho, p = spearmanr(ok["gt_pct"], ok["delta"])
print(f"\n  Spearman rho(Δ GT %, Δ mAP) = {rho:+.3f}   p = {p:.4f}   n = {len(ok)}")
print("  Positive rho supports DEC-124: the classes whose annotations were most incomplete")
print("  are the classes that improved most once the annotations were completed.")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.scatter(gt["gt_pct"], gt["delta"], s=70, color=BLUE, edgecolor=SURFACE, linewidth=1.5, zorder=3)
# selective labels only, alternating above/below in x-order so neighbours never collide
lab = (gt[(gt["delta"].abs() >= 0.02) | (gt["gt_pct"].abs() >= 25)]
       .sort_values("gt_pct").reset_index(drop=True))
for i, r in lab.iterrows():
    ax.annotate(r["class"], (r["gt_pct"], r["delta"]), textcoords="offset points",
                xytext=(8, 7 if i % 2 == 0 else -13), fontsize=8.5, color=INK_2)
ax.margins(x=0.12)
ax.axhline(0, color=GRID, lw=1.2); ax.axvline(0, color=GRID, lw=1.2)
ax.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
style_axes(ax, xlabel="change in ground-truth instances (%)", ylabel="Δ mAP@0.5 (v2 − v1)",
           title=f"Annotation recovery vs accuracy gain  ·  Spearman ρ = {rho:+.3f}")
fig.tight_layout(); fig.savefig(FIGS / "gt_recovery_vs_gain_v2.png", dpi=160,
                                facecolor=SURFACE); plt.show()

## 5. Training progress

Two panels rather than one with twin y-axes: mAP and loss are different scales, and a dual-axis
chart lets the reader infer a crossing point that does not exist.

In [ ]:
fit = 0.1 * hist["metrics/mAP50(B)"] + 0.9 * hist["metrics/mAP50-95(B)"]
best_ep = int(hist.loc[fit.idxmax(), "epoch"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

ax = axes[0]
ax.plot(hist["epoch"], hist["metrics/mAP50(B)"],    color=BLUE,   lw=2, label="mAP@0.5")
ax.plot(hist["epoch"], hist["metrics/mAP50-95(B)"], color=ORANGE, lw=2, label="mAP@0.5:0.95")
ax.axvline(best_ep, color=GRID, lw=1.5, ls="--", zorder=0)
ax.annotate(f"best epoch {best_ep}", (best_ep, ax.get_ylim()[1]), textcoords="offset points",
            xytext=(6, -13), fontsize=8.5, color=INK_2, va="top")   # top: clears the legend
ax.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
style_axes(ax, xlabel="epoch", ylabel="mAP (val split)", title="Validation accuracy")
ax.legend(frameon=False, fontsize=9, loc="lower right", labelcolor=INK_2)

ax = axes[1]
for col, c, lab in [("train/box_loss", BLUE, "box (train)"), ("val/box_loss", BLUE_LT, "box (val)"),
                    ("train/cls_loss", ORANGE, "cls (train)"), ("val/cls_loss", "#f5b394", "cls (val)")]:
    if col in hist: ax.plot(hist["epoch"], hist[col], color=c, lw=2, label=lab)
ax.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
style_axes(ax, xlabel="epoch", ylabel="loss", title="Losses")
ax.legend(frameon=False, fontsize=9, ncol=2, labelcolor=INK_2)

fig.tight_layout(); fig.savefig(FIGS / "training_curves_v2.png", dpi=160,
                                facecolor=SURFACE); plt.show()

print(f"  epochs recorded     {len(hist)}")
print(f"  best epoch          {best_ep}  (ultralytics fitness = 0.1·mAP50 + 0.9·mAP50-95)")
print(f"  final-epoch mAP50   {hist['metrics/mAP50(B)'].iloc[-1]:.4f} (val)")
print(f"  best     mAP50      {hist['metrics/mAP50(B)'].max():.4f} (val)")
print(f"  test-split mAP50    {summary['map50']:.4f}  <- the reported figure")

Two features of the loss panel are worth naming, because both look like faults and neither is:

**The step down at epoch 90.** Ultralytics disables mosaic augmentation for the final 10 epochs
(`close_mosaic=10`). Training loss drops sharply because the task itself gets easier, not because
the model suddenly improved — validation accuracy barely moves across the same boundary.

**Validation loss turns upward around epoch 50 while training loss keeps falling.** That is mild
overfitting, and it is why the best epoch is 78 rather than 100. `patience` is set to 100000 —
early stopping is deliberately disabled (**DEC-120**) so that v1 and v2 run the identical
schedule; ultralytics still keeps `best.pt` from the best epoch, which is the checkpoint
evaluated throughout this report. The gap between the best val mAP@0.5 (0.6958) and the final
epoch's (0.6893) is the size of that effect.

## 6. Pre-registered criteria (DEC-122)

**DEC-122** fixed the criterion *before* v1 ran: a primary target of mAP@0.5 = 0.70, per-class
floors on the five road-safety classes, and Pole and Potholes named in advance as expected-weak
so that a low score on them could not be reinterpreted after the fact.

**v2 clears the primary target that v1 missed.** One floor still fails — and it is worth being
precise about which, because the earlier assumption was that Pole would be the culprit.

In [ ]:
print(f"  primary target  mAP@0.5 >= {TARGET:.2f}")
print(f"    v1  {summary_v1['map50']:.4f}   {'PASS' if summary_v1['primary_pass'] else 'FAIL'}")
print(f"    v2  {summary['map50']:.4f}   {'PASS' if summary['primary_pass'] else 'FAIL'}")

v1_floor = {f["class"]: f["map50"] for f in summary_v1["per_class_floors"]}
print(f"\n  per-class floors          {'floor':>7} {'v1':>8} {'v2':>8} {'Δ':>8}   result")
print("  " + "-" * 62)
for f in summary["per_class_floors"]:
    a = v1_floor.get(f["class"], float("nan"))
    print(f"    {f['class']:<20} {f['floor']:>7.2f} {a:>8.4f} {f['map50']:>8.4f} "
          f"{f['map50']-a:>+8.4f}   {'PASS' if f['pass'] else 'FAIL'}")

failed = [f for f in summary["per_class_floors"] if not f["pass"]]
print(f"\n  floors_pass = {summary['floors_pass']}  ({len(failed)} of "
      f"{len(summary['per_class_floors'])} floors failed)")
for f in failed:
    print(f"    {f['class']}: {f['map50']:.5f} vs floor {f['floor']:.2f} "
          f"- short by {f['floor']-f['map50']:.5f}")
print(f"\n  expected-weak (named in advance, not failures): "
      f"{', '.join(summary['expected_weak'])}")
for c in summary["expected_weak"]:
    r = df[df['class'] == c].iloc[0]
    print(f"    {c:<10} mAP50 {r['mAP50']:.4f}   recall {r['recall']:.4f}")

> **Read the remaining floor failure carefully.** Person reaches 0.7414 against a 0.75 floor —
> short by 0.0086 — having risen from 0.5226 on the same labels. The floor was pre-registered
> against v1-era ground truth, which contained roughly half as many Person boxes; v2 is being
> held to that floor on a substantially harder and more complete test set. Pole and Potholes,
> the two classes long assumed to be the problem, carry **no floor at all** — DEC-122 listed
> them as expected-weak precisely so they could not be treated as failures after the fact.

## 7. Error budget

Raw false-positive counts are not comparable across v1 and v2 — v2's test ground truth contains
more boxes, so both errors and opportunities grew. Normalising by ground-truth instances is what
makes the two runs commensurable.

In [ ]:
fp_v2, gt_v2 = summary["false_positives_vs_background"], summary["ground_truth_instances"]
v1_14row = df_v1[df_v1["class"] != "Shelf"]
fp_v1_14, gt_v1_14 = v1_14row["false_positives_vs_background"].sum(), v1_14row["instances"].sum()

print(f"  {'':<26}{'v1 (14 cls)':>13}{'v2':>10}")
print("  " + "-" * 49)
print(f"  {'GT instances':<26}{gt_v1_14:>13,}{gt_v2:>10,}")
print(f"  {'FP vs background':<26}{fp_v1_14:>13,}{fp_v2:>10,}")
print(f"  {'FP per GT instance':<26}{fp_v1_14/gt_v1_14:>13.3f}{fp_v2/gt_v2:>10.3f}")
print(f"  {'missed detections':<26}{v1_14row['missed_detections'].sum():>13,}"
      f"{df['missed_detections'].sum():>10,}")
print(f"  {'missed per GT instance':<26}"
      f"{v1_14row['missed_detections'].sum()/gt_v1_14:>13.3f}"
      f"{df['missed_detections'].sum()/gt_v2:>10.3f}")

d = df.sort_values("instances", ascending=False).reset_index(drop=True)
y, h = range(len(d)), 0.38
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh([i + h/2 + 0.01 for i in y], d["missed_detections"], height=h,
        color=BLUE,   label="missed detections")
ax.barh([i - h/2 - 0.01 for i in y], d["false_positives_vs_background"], height=h,
        color=ORANGE, label="false positives vs background")
ax.set_yticks(list(y)); ax.set_yticklabels(d["class"], fontsize=9, color=INK)
ax.invert_yaxis()
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
style_axes(ax, xlabel="boxes (test split)", title="v2 error budget, classes by ground-truth frequency")
ax.legend(frameon=False, fontsize=9, loc="lower right", labelcolor=INK_2)
fig.tight_layout(); fig.savefig(FIGS / "error_budget_v2.png", dpi=160,
                                facecolor=SURFACE); plt.show()

## 8. Summary table

In [ ]:
out = (df[["class", "instances", "images", "precision", "recall", "f1", "mAP50", "mAP50_95"]]
       .merge(shared[["class", "v1_on_v2a", "delta"]], on="class", how="left")
       .rename(columns={"v1_on_v2a": "v1_mAP50", "delta": "delta_mAP50"})
       .sort_values("mAP50", ascending=False)
       .round(4))
out.to_csv(FIGS / "per_class_table_v2.csv", index=False)
print(out.to_string(index=False))
print(f"\n  written: {(FIGS / 'per_class_table_v2.csv').relative_to(ROOT)}")
print(f"  figures: {', '.join(sorted(p.name for p in FIGS.glob('*.png')))}")

## 9. What this run does and does not establish

**Establishes.** On a frozen image set, frozen splits and an identical training schedule,
completing and de-duplicating the annotations moved mAP@0.5 from 0.6825 to 0.7115, **+0.0289**,
and carried the model past the 0.70 target pre-registered in **DEC-122**. The gain is
concentrated exactly where **DEC-124** predicted — Person, whose ground truth nearly doubled.

**A claim to avoid — it does not survive scrutiny.** It is tempting to note that v1's own score
*rose* from 0.6744 to 0.6825 when re-scored on the deduplicated labels, and to read that as
independent validation of **DEC-132** and **DEC-133**. **That inference does not hold.** Those two
figures differ in *two* ways at once: the labels changed, but the measurement also moved from a
laptop to the training pod — and **DEC-134** measures that path difference alone at roughly
**+0.007**, against a total gap of **+0.0081**. The dedup effect cannot be separated from the
measurement effect here and must not be reported as if it could.

**Only one comparison in this project is fully controlled**, and it is the headline: v2 0.7115 vs
v1 0.6825 — same pod, same current labels, same 14 classes, same NMS settings, same images. Every
other v1 figure in circulation (0.6695, 0.6417, 0.6744, 0.7037) carries a different confound, and
none of them may be chained into a narrative.

**Does not establish.** That the architecture was ever the limiting factor. **Both v1 and v2
trained `yolov8s.pt`** — the backbone was never varied, so nothing here speaks to capacity.
RSUD20K measured YOLOv8-M at 71.8 against YOLOv8-S at 69.4 (**+2.4**) in a comparable domain,
which suggests roughly this much again may be available from capacity alone. Two limits on that
reading: their YOLOv8-L scored **70.4, *worse* than M**, so the trend does not extrapolate past
M; and **their paper does not state its IoU threshold — @0.5 is inferred**, a caveat DEC-124
requires to travel with any comparison to their numbers.

*(Unrelated, because the filename invites confusion: `notebooks/yolov8m.pt` is the COCO proxy
that pre-populates box suggestions in the FiftyOne review overlay per **DEC-106**. It is
annotation scaffolding — never trained, never evaluated, never deployed.)*

**A methodological caveat worth stating in the write-up.** Two defects reached the training pool
having passed **every automated gate** — the one-class id shift of **DEC-130** and the 824
duplicate boxes of **DEC-133**. Both were caught only by a human looking at boxes in FiftyOne. A
shifted or duplicated label is structurally valid: ids in range, counts plausible, dataset checks
green. That is a finding about merged-dataset pipelines in its own right, not a footnote.